In [ ]:
"""
=============================================================================
BORING BAR CAVITY OPTIMISATION — PARAMETERS 2 AND 3
=============================================================================

PARAMETER 2 : Cavity length   L4   [m]
              Range : 0.040 – 0.110 m   (40 – 110 mm)
              Effect: longer cavity → heavier damper → larger mass ratio μ
                      but removes more stiff material → lower fn1

PARAMETER 3 : Cavity inner diameter   di_cav   [m]
              Equiv : wall thickness  wall = (D_main – di_cav)/2
              Range : di_cav  0.026 – 0.044 m  (wall  12 – 3 mm)
              Effect: larger bore → heavier damper → larger μ
                      but thinner wall → lower section-4 bending stiffness

FIXED  (from Parameter-1 optimum)
──────────────────────────────────
  L3 = 0.100 m   (cavity placed at 170 mm from the fixed end)

GEOMETRY — 5 sections (same as Parameter 1)
  Sec 1 [0,       L1]     : hollow flange,  D1=63 mm, di=20 mm
  Sec 2 [L1,      L1+L2]  : taper,          D1 → D_main,  di=20 mm
  Sec 3 [L1+L2,   L1+L2+L3]: hollow shaft,  D_main=50 mm, di=20 mm
  Sec 4 [L1+L2+L3,  …+L4] : cavity,         D_main=50 mm, di=di_cav  ← swept
  Sec 5 […,       L_total] : solid tip,      D_main=50 mm  (L5 adapts)

PHYSICS
  • 5-section Euler-Bernoulli beam characteristic equation (20×20)
  • Single-mode modal projection with all rotational effects (Eq. 28)
    – Coriolis, Gyroscopic, Centrifugal, Rotational-inertia
  • Speed-dependent FRF at each stability pocket (non-zero-speed method)
  • Budak-Altintas pocket-speed stability lobe algorithm
  • Den Hartog absorber tuning at every (L4, di_cav) point

RESULTS PROVIDED
  • fn1, Mm, PSI(zd), md, μ, kd_DH, cd_DH at each grid point
  • Minimum stable depth of cut: no-absorber vs DH-absorber
  • Improvement ratio heat-maps and 3-D surfaces
  • Full stability lobe diagrams at best, worst and reference designs
  • Natural frequency vs spindle speed at best design
  • FRF comparison (direct + cross) at best design
  • Individual rotational-force contribution to stability
=============================================================================
"""

import numpy as np
from scipy.optimize import brentq
from scipy.linalg import svd
from scipy.integrate import quad
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 1.  CONSTANTS AND GEOMETRY
# ═══════════════════════════════════════════════════════════════════════════

E_body   = 280e9           # Pa   body material
rho_body = 7850.0 * 1.2    # kg/m³
rho_WC   = 14_500.0        # kg/m³  tungsten-carbide damper
zeta_str = 0.01            # structural modal damping ratio

# Fixed overall dimensions
L_total  = 0.300;  D1_out = 0.063;  D_main = 0.050
di_body  = 0.020;  L1 = 0.040;  L2 = 0.030
L3_OPT   = 0.100              # optimum from Parameter-1 study
L5_MIN   = 0.015              # minimum solid-tip length
GAP_RAD  = 0.001              # 1 mm radial gap damper ↔ cavity wall
CLR_AX   = 0.010              # 10 mm total axial clearance (5 mm each end)
WALL_MIN = 0.003              # 3 mm minimum wall thickness

# Cutting parameters
Nt       = 2                  # inserts (boring bar)
Ktc      = 2000e6             # Pa   tangential cutting coeff
Krc      = 500e6              # Pa   radial cutting coeff
ratio_kr = Krc / Ktc          # 0.25

# Sweep ranges
N_L4     = 8                  # number of L4 values
N_DI     = 8                  # number of di_cav values
L4_vals  = np.linspace(0.040, 0.110, N_L4)          # 40–110 mm
di_cav_vals = np.linspace(0.026, 0.044, N_DI)       # 26–44 mm

# Speed grid for stability lobes
N_grid   = np.linspace(100, 6000, 400)
speeds_spd = np.linspace(0, 6000, 200)

#SAVE     = "/mnt/user-data/outputs"

print("=" * 70)
print("BORING BAR — PARAMETER 2 (cavity length) & 3 (cavity diameter) SWEEP")
print("=" * 70)
print(f"  L3 fixed = {L3_OPT*1e3:.0f} mm   (from Parameter-1 optimum)")
print(f"  L4 range = {L4_vals[0]*1e3:.0f} – {L4_vals[-1]*1e3:.0f} mm  ({N_L4} values)")
print(f"  di_cav   = {di_cav_vals[0]*1e3:.0f} – {di_cav_vals[-1]*1e3:.0f} mm  ({N_DI} values)")
print(f"  Grid     = {N_L4}×{N_DI} = {N_L4*N_DI} configurations")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 2.  CROSS-SECTION HELPERS
# ═══════════════════════════════════════════════════════════════════════════

def xsec(D_out, d_in=0.0):
    """Return (A, I, Ip) for circular cross-section."""
    ro, ri = D_out / 2, d_in / 2
    A  = np.pi * (ro**2  - ri**2)
    I  = np.pi * (ro**4  - ri**4) / 4
    Ip = np.pi * (ro**4  - ri**4) / 2
    return A, I, Ip


# Pre-compute fixed sections 1, 2-avg, 3, 5
A1s, I1s, Ip1s = xsec(D1_out, di_body)
A5s, I5s, Ip5s = xsec(D_main,  0.0)
A3s, I3s, Ip3s = xsec(D_main, di_body)
_fr = np.linspace(0, 1, 300)
_Dv = D1_out + (D_main - D1_out) * _fr
A2s  = np.mean([xsec(d, di_body)[0] for d in _Dv])
I2s  = np.mean([xsec(d, di_body)[1] for d in _Dv])
Ip2s = np.mean([xsec(d, di_body)[2] for d in _Dv])

# Basis functions for Euler-Bernoulli beam
def P(b, z):
    return np.array([np.cos(b*z),  np.cosh(b*z),  np.sin(b*z),  np.sinh(b*z)])
def dP(b, z):
    return np.array([-b*np.sin(b*z), b*np.sinh(b*z), b*np.cos(b*z), b*np.cosh(b*z)])
def d2P(b, z):
    return np.array([-b**2*np.cos(b*z),  b**2*np.cosh(b*z),
                     -b**2*np.sin(b*z),  b**2*np.sinh(b*z)])
def d3P(b, z):
    return np.array([b**3*np.sin(b*z),   b**3*np.sinh(b*z),
                    -b**3*np.cos(b*z),   b**3*np.cosh(b*z)])

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 3.  5-SECTION CHARACTERISTIC MATRIX & NATURAL FREQUENCY
# ═══════════════════════════════════════════════════════════════════════════

def make_section_arrays(di_cav, L4):
    """Build per-section EI, rhoA, rhoI, rhoIp arrays for given cavity params."""
    A4, I4, Ip4 = xsec(D_main, di_cav)
    L5 = L_total - L1 - L2 - L3_OPT - L4
    EI_  = [E_body*I1s,  E_body*I2s,  E_body*I3s,  E_body*I4,  E_body*I5s]
    rhoA_= [rho_body*A1s, rho_body*A2s, rho_body*A3s, rho_body*A4, rho_body*A5s]
    rhoI_= [rho_body*I1s, rho_body*I2s, rho_body*I3s, rho_body*I4, rho_body*I5s]
    rhoIp=[rho_body*Ip1s,rho_body*Ip2s,rho_body*Ip3s,rho_body*Ip4,rho_body*Ip5s]
    c_   = np.sqrt(np.array(EI_) / np.array(rhoA_))
    zb   = [0, L1, L1+L2, L1+L2+L3_OPT, L1+L2+L3_OPT+L4, L_total]
    return EI_, rhoA_, rhoI_, rhoIp, c_, zb, L5


def char_matrix_5s(omega, EI_, c_, zb):
    """20×20 characteristic matrix for 5-section beam."""
    bs  = np.sqrt(omega / c_)
    M   = np.zeros((20, 20))
    r   = 0
    # Fixed-end BCs
    M[r, 0:4]  = P(bs[0], zb[0]);   r += 1
    M[r, 0:4]  = dP(bs[0], zb[0]);  r += 1
    # Free-end BCs
    M[r,16:20] = EI_[4] * d2P(bs[4], zb[5]);  r += 1
    M[r,16:20] = EI_[4] * d3P(bs[4], zb[5]);  r += 1
    # Interface conditions
    for k in range(4):
        z  = zb[k+1]
        bl = bs[k];  br = bs[k+1];  EIl = EI_[k];  EIr = EI_[k+1]
        cl = 4*k;    cr = 4*(k+1)
        M[r, cl:cl+4] =      P(bl,z); M[r, cr:cr+4] =     -P(br,z); r += 1
        M[r, cl:cl+4] =     dP(bl,z); M[r, cr:cr+4] =    -dP(br,z); r += 1
        M[r, cl:cl+4] = EIl*d2P(bl,z); M[r, cr:cr+4]=-EIr*d2P(br,z); r += 1
        M[r, cl:cl+4] = EIl*d3P(bl,z); M[r, cr:cr+4]=-EIr*d3P(br,z); r += 1
    return M


def sdet_sign(omega, EI_, c_, zb):
    try:
        s, _ = np.linalg.slogdet(char_matrix_5s(omega, EI_, c_, zb))
        return float(s)
    except Exception:
        return 0.0


def find_fn1_and_mode(di_cav, L4, n_scan=80_000):
    """
    For given cavity parameters compute:
      fn1, Mm, Rii, Gii, PSI_zd, PSI_zf, zb, wn1, md, mu, kd_DH, cd_DH
    Returns None if L5 < L5_MIN or no root found.
    """
    EI_, rhoA_, rhoI_, rhoIp_, c_, zb, L5 = make_section_arrays(di_cav, L4)
    if L5 < L5_MIN:
        return None

    # Scan for first zero of det-sign
    scan  = np.linspace(300.0, 50_000.0, n_scan)
    signs = np.array([sdet_sign(om, EI_, c_, zb) for om in scan])
    wn1   = None
    for i in range(len(signs) - 1):
        if signs[i] * signs[i+1] < 0 and signs[i] != 0 and signs[i+1] != 0:
            try:
                wn1 = brentq(lambda o: sdet_sign(o, EI_, c_, zb),
                             scan[i], scan[i+1], xtol=0.5)
                break
            except Exception:
                pass
    if wn1 is None:
        return None

    # Mode-shape coefficients from SVD null vector
    _, _, Vh = svd(char_matrix_5s(wn1, EI_, c_, zb))
    C_  = Vh[-1]
    bs  = np.sqrt(wn1 / c_)

    def _raw(z):
        if   z <= zb[1]: return C_[ 0: 4] @ P(bs[0], z)
        elif z <= zb[2]: return C_[ 4: 8] @ P(bs[1], z)
        elif z <= zb[3]: return C_[ 8:12] @ P(bs[2], z)
        elif z <= zb[4]: return C_[12:16] @ P(bs[3], z)
        else:            return C_[16:20] @ P(bs[4], z)

    def _draw(z):
        if   z <= zb[1]: return C_[ 0: 4] @ dP(bs[0], z)
        elif z <= zb[2]: return C_[ 4: 8] @ dP(bs[1], z)
        elif z <= zb[3]: return C_[ 8:12] @ dP(bs[2], z)
        elif z <= zb[4]: return C_[12:16] @ dP(bs[3], z)
        else:            return C_[16:20] @ dP(bs[4], z)

    sc   = _raw(L_total)
    psi  = lambda z, _r=_raw,  _s=sc: _r(z) / _s
    dpsi = lambda z, _d=_draw, _s=sc: _d(z) / _s

    # Modal integrals over all 5 sections
    Mm  = sum(quad(lambda z: rhoA_[k] * psi(z)**2,  zb[k], zb[k+1], limit=150)[0]
              for k in range(5))
    Rii = sum(quad(lambda z: rhoI_[k] * dpsi(z)**2, zb[k], zb[k+1], limit=150)[0]
              for k in range(5))
    Gii = sum(quad(lambda z: rhoIp_[k]* dpsi(z)**2, zb[k], zb[k+1], limit=150)[0]
              for k in range(5))

    zd     = (zb[3] + zb[4]) / 2.0
    PSI_zd = psi(zd)
    PSI_zf = psi(L_total)       # = 1 by normalisation
    Km     = wn1**2 * Mm
    Cm_str = 2.0 * zeta_str * wn1 * Mm

    # Damper mass (WC cylinder)
    d_damp = di_cav - 2 * GAP_RAD
    L_damp = L4 - CLR_AX
    md     = rho_WC * np.pi * (d_damp / 2)**2 * L_damp

    # Den Hartog tuning
    mu     = md / Mm
    wd_dh  = wn1 / (1.0 + mu)
    zeta_dh= np.sqrt(3.0 * mu / (8.0 * (1.0 + mu)))
    kd_dh  = md * wd_dh**2
    cd_dh  = 2.0 * zeta_dh * md * wd_dh

    return dict(
        fn1=wn1/(2*np.pi), wn1=wn1, Mm=Mm, Rii=Rii, Gii=Gii, Km=Km,
        Cm_str=Cm_str, PSI_zd=PSI_zd, PSI_zf=PSI_zf, zd=zd, zb=zb,
        psi=psi, dpsi=dpsi,
        md=md, mu=mu, kd_dh=kd_dh, cd_dh=cd_dh, zeta_dh=zeta_dh,
        L5=L5, di_cav=di_cav, L4=L4,
        EI_=EI_, rhoA_=rhoA_
    )

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 4.  SYSTEM MATRICES (single-mode, all rotational effects, Eq. 28)
# ═══════════════════════════════════════════════════════════════════════════

def build_4x4(Omega, r):
    """4×4 matrices for beam + absorber with all rotational forces."""
    Mm=r['Mm']; Rii=r['Rii']; Gii=r['Gii']
    Km=r['Km']; Cm=r['Cm_str']; pzd=r['PSI_zd']
    kd=r['kd_dh']; cd=r['cd_dh']; md=r['md']

    M11   = Mm + Rii
    K11   = Km - Omega**2 * Mm
    K_abs = kd - Omega**2 * md
    Cor   = 2*Omega*Mm;  Gyr = 2*Omega*Gii
    C_off = Cor - Gyr
    Ca    = 2*Omega*md

    M = np.diag([M11, md, M11, md])
    K = np.array([[K11+kd*pzd**2, -kd*pzd, 0, 0],
                  [-kd*pzd,        K_abs,   0, 0],
                  [0, 0, K11+kd*pzd**2, -kd*pzd],
                  [0, 0, -kd*pzd,        K_abs]])
    D = np.array([[Cm+cd*pzd**2, -cd*pzd,  C_off, 0],
                  [-cd*pzd,       cd,       0,     Ca],
                  [-C_off, 0, Cm+cd*pzd**2, -cd*pzd],
                  [0, -Ca, -cd*pzd,          cd]])
    return M, D, K


def build_2x2(Omega, r):
    """2×2 matrices for beam WITHOUT absorber."""
    Mm=r['Mm']; Rii=r['Rii']; Gii=r['Gii']; Km=r['Km']; Cm=r['Cm_str']
    M11 = Mm + Rii
    K11 = Km - Omega**2 * Mm
    Cof = 2*Omega*Mm - 2*Omega*Gii
    M   = np.diag([M11, M11])
    K   = np.diag([K11, K11])
    D   = np.array([[Cm, Cof], [-Cof, Cm]])
    return M, D, K

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 5.  FRF, DIRECTIONAL COEFFICIENTS, STABILITY
# ═══════════════════════════════════════════════════════════════════════════

def frf_at_wc(wc, M, D, K, PSI_zf):
    n = M.shape[0];  nb = n // 2
    fx = np.zeros(n);  fx[0]  = PSI_zf
    fy = np.zeros(n);  fy[nb] = PSI_zf
    Z  = -wc**2*M + 1j*wc*D + K
    try:
        H = np.linalg.solve(Z, np.column_stack([fx, fy]))
        return PSI_zf*H[0, 0], PSI_zf*H[0, 1]
    except np.linalg.LinAlgError:
        return 0j, 0j


def dir_coeffs(case="up50"):
    phi1, phi2 = (0, np.pi/2) if case == "up50" else (0, np.pi)
    phi = np.linspace(phi1, phi2, 80_000)
    r   = ratio_kr
    return (np.trapezoid(np.sin(phi)*(-np.cos(phi)-r*np.sin(phi)), phi),
            np.trapezoid(np.cos(phi)*(-np.cos(phi)-r*np.sin(phi)), phi),
            np.trapezoid(np.sin(phi)*(np.sin(phi) -r*np.cos(phi)), phi),
            np.trapezoid(np.cos(phi)*(np.sin(phi) -r*np.cos(phi)), phi))

AXX_UP, AXY_UP, AYX_UP, AYY_UP = dir_coeffs("up50")


def stability_lobes(r, speed_dep=True, wc_pts=4000, max_lobe=20,
                    Nmin=100.0, Nmax=6100.0, alim_cap=0.020):
    """
    Full pocket-speed stability lobe cloud.
    Returns (N_rpm, alim_mm, fc_hz).
    """
    axx, axy, ayx, ayy = AXX_UP, AXY_UP, AYX_UP, AYY_UP
    wn1    = r['wn1']
    PSI_zf = r['PSI_zf']
    wc_arr = np.linspace(0.3*wn1, 2.5*wn1, wc_pts)
    M0, D0, K0 = build_4x4(0.0, r)

    N_out = [];  a_out = [];  fc_out = []
    for wc in wc_arr:
        fc = wc / (2*np.pi)
        hxx, hxy = frf_at_wc(wc, M0, D0, K0, PSI_zf)
        hyy = hxx;  hyx = hxy
        h11=axx*hxx+axy*hyx; h12=axx*hxy+axy*hyy
        h21=ayx*hxx+ayy*hyx; h22=ayx*hxy+ayy*hyy
        tr=h11+h22; dv=h11*h22-h12*h21
        if abs(dv) < 1e-60:
            continue
        disc=tr**2-4*dv; sq=np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*dv), (-tr-sq)/(2*dv)]:
            rl=np.real(lam); il=np.imag(lam)
            if rl >= 0:
                continue
            al = -2*np.pi*rl/(Nt*Ktc)*(1+(il/rl)**2)
            if al <= 0 or al > alim_cap:
                break
            kappa   = il / rl
            epsilon = np.pi - 2*np.arctan(kappa)
            for k in range(max_lobe+1):
                denom = k + epsilon/(2*np.pi)
                if denom <= 0:
                    continue
                Np = 60.0*fc / (Nt*denom)
                if not (Nmin <= Np <= Nmax):
                    continue
                if speed_dep:
                    Om = Np*2*np.pi/60
                    Mp,Dp,Kp = build_4x4(Om, r)
                    hxx2,hxy2 = frf_at_wc(wc,Mp,Dp,Kp,PSI_zf)
                    hyy2=hxx2; hyx2=hxy2
                    h11_=axx*hxx2+axy*hyx2; h12_=axx*hxy2+axy*hyy2
                    h21_=ayx*hxx2+ayy*hyx2; h22_=ayx*hxy2+ayy*hyy2
                    tr_=h11_+h22_; dv_=h11_*h22_-h12_*h21_
                    if abs(dv_)<1e-60: continue
                    disc_=tr_**2-4*dv_; sq_=np.sqrt(disc_+0j)
                    for lam_ in [(-tr_+sq_)/(2*dv_),(-tr_-sq_)/(2*dv_)]:
                        rl_=np.real(lam_); il_=np.imag(lam_)
                        if rl_ < 0:
                            al_=-2*np.pi*rl_/(Nt*Ktc)*(1+(il_/rl_)**2)
                            if al_ > 0:
                                kap2=il_/rl_; eps2=np.pi-2*np.arctan(kap2)
                                den2=k+eps2/(2*np.pi)
                                if den2>0:
                                    N2=60*fc/(Nt*den2)
                                    if Nmin<=N2<=Nmax:
                                        N_out.append(N2)
                                        a_out.append(al_*1000)
                                        fc_out.append(fc)
                            break
                else:
                    N_out.append(Np); a_out.append(al*1000); fc_out.append(fc)
            break
    return np.array(N_out), np.array(a_out), np.array(fc_out)


def lower_env(N, a, Ng, win=100.0):
    env = np.full(len(Ng), np.nan)
    for i, Ngi in enumerate(Ng):
        mask = np.abs(N - Ngi) < win
        if mask.sum() > 0:
            env[i] = a[mask].min()
    return env


def min_alim(r, speed_dep=False):
    """Quick minimum alim from FRF sweep (no full lobe — faster)."""
    PSI_zf = r['PSI_zf']
    wn1    = r['wn1']
    axx,axy,ayx,ayy = AXX_UP, AXY_UP, AYX_UP, AYY_UP
    M0,D0,K0 = build_4x4(0.0, r)
    al_min = np.inf
    for fc in np.linspace(0.3*wn1/(2*np.pi), 2.5*wn1/(2*np.pi), 2000):
        wc=2*np.pi*fc
        hxx,hxy=frf_at_wc(wc,M0,D0,K0,PSI_zf)
        hyy=hxx; hyx=hxy
        h11=axx*hxx+axy*hyx; h12=axx*hxy+axy*hyy
        h21=ayx*hxx+ayy*hyx; h22=ayx*hxy+ayy*hyy
        tr=h11+h22; dv=h11*h22-h12*h21
        if abs(dv)<1e-60: continue
        disc=tr**2-4*dv; sq=np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*dv),(-tr-sq)/(2*dv)]:
            rl=np.real(lam); il=np.imag(lam)
            if rl<0:
                al=-2*np.pi*rl/(Nt*Ktc)*(1+(il/rl)**2)
                if 0<al<al_min: al_min=al
    return al_min if al_min != np.inf else 0.0


def min_alim_noabs(r):
    PSI_zf=r['PSI_zf']; wn1=r['wn1']
    axx,axy,ayx,ayy=AXX_UP,AXY_UP,AYX_UP,AYY_UP
    M0,D0,K0=build_2x2(0.0,r)
    al_min=np.inf
    for fc in np.linspace(0.3*wn1/(2*np.pi), 2.5*wn1/(2*np.pi), 2000):
        wc=2*np.pi*fc
        hxx,hxy=frf_at_wc(wc,M0,D0,K0,PSI_zf)
        hyy=hxx; hyx=hxy
        h11=axx*hxx+axy*hyx; h12=axx*hxy+axy*hyy; h21=ayx*hxx+ayy*hyx; h22=ayx*hxy+ayy*hyy
        tr=h11+h22; dv=h11*h22-h12*h21
        if abs(dv)<1e-60: continue
        disc=tr**2-4*dv; sq=np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*dv),(-tr-sq)/(2*dv)]:
            rl=np.real(lam); il=np.imag(lam)
            if rl<0:
                al=-2*np.pi*rl/(Nt*Ktc)*(1+(il/rl)**2)
                if 0<al<al_min: al_min=al
    return al_min if al_min != np.inf else 0.0

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.  NATURAL FREQUENCY vs SPEED
# ═══════════════════════════════════════════════════════════════════════════

def nat_freqs_vs_speed(r, speeds_rpm):
    fw=[]; bw=[]; ab=[]
    for rpm in speeds_rpm:
        Om=rpm*2*np.pi/60
        M,D,K=build_4x4(Om,r)
        sz=M.shape[0]
        A=np.zeros((2*sz,2*sz)); Mi=np.linalg.inv(M)
        A[:sz,:sz]=-Mi@D; A[:sz,sz:]=-Mi@K; A[sz:,:sz]=np.eye(sz)
        evs=np.linalg.eigvals(A)
        fs=sorted(np.abs(evs.imag)/(2*np.pi)); fs=[f for f in fs if f>5]
        dedup=[]
        for f in fs:
            if not dedup or abs(f-dedup[-1])>0.5: dedup.append(f)
        wn_hz=r['wn1']/(2*np.pi)
        beam=[f for f in dedup if 0.3*wn_hz<f<3.5*wn_hz]
        abso=[f for f in dedup if abs(f-r['kd_dh']**0.5/(2*np.pi*r['md']**0.5))<300]
        if len(beam)>=2: bw.append(beam[0]); fw.append(beam[-1])
        elif len(beam)==1: bw.append(beam[0]); fw.append(beam[0])
        else: bw.append(np.nan); fw.append(np.nan)
        ab.append(abso[0] if abso else np.nan)
    return np.array(fw), np.array(bw), np.array(ab)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 7.  PARAMETER SWEEP
# ═══════════════════════════════════════════════════════════════════════════

# Result matrices (indexed [L4_idx, di_idx])
fn1_mat   = np.full((N_L4, N_DI), np.nan)
Mm_mat    = np.full((N_L4, N_DI), np.nan)
PSI_mat   = np.full((N_L4, N_DI), np.nan)
md_mat    = np.full((N_L4, N_DI), np.nan)
mu_mat    = np.full((N_L4, N_DI), np.nan)
kd_mat    = np.full((N_L4, N_DI), np.nan)
cd_mat    = np.full((N_L4, N_DI), np.nan)
alim_na_mat=np.full((N_L4, N_DI), np.nan)
alim_dh_mat=np.full((N_L4, N_DI), np.nan)
ratio_mat  = np.full((N_L4, N_DI), np.nan)
res_store  = [[None]*N_DI for _ in range(N_L4)]

print(f"\n{'L4':>6} {'di_cav':>8} {'wall':>6} {'fn1':>7} {'Mm':>7} "
      f"{'PSI_zd':>7} {'md':>7} {'mu':>6} {'kd[MN/m]':>9} {'cd':>7} "
      f"{'al_na[mm]':>10} {'al_dh[mm]':>10} {'ratio':>6}")
print("-"*105)

for i, L4 in enumerate(L4_vals):
    for j, di_cav in enumerate(di_cav_vals):
        wall = (D_main - di_cav) / 2
        if wall < WALL_MIN - 1e-6:
            continue
        r = find_fn1_and_mode(di_cav, L4)
        if r is None:
            print(f"  {L4*1e3:5.0f}  {di_cav*1e3:8.0f}  {wall*1e3:5.1f}  SKIP (L5 too short or no root)")
            continue

        al_na = min_alim_noabs(r)
        al_dh = min_alim(r)
        ratio = al_dh / al_na if al_na > 0 else 0.0

        fn1_mat[i,j]=r['fn1'];  Mm_mat[i,j]=r['Mm']
        PSI_mat[i,j]=r['PSI_zd']; md_mat[i,j]=r['md']
        mu_mat[i,j]=r['mu'];   kd_mat[i,j]=r['kd_dh']/1e6
        cd_mat[i,j]=r['cd_dh']; alim_na_mat[i,j]=al_na*1e3
        alim_dh_mat[i,j]=al_dh*1e3; ratio_mat[i,j]=ratio
        res_store[i][j] = r

        print(f"  {L4*1e3:5.0f}  {di_cav*1e3:8.0f}  {wall*1e3:5.1f}  "
              f"{r['fn1']:7.1f}  {r['Mm']:7.4f}  {r['PSI_zd']:7.4f}  "
              f"{r['md']:7.4f}  {r['mu']:6.3f}  {r['kd_dh']/1e6:9.4f}  "
              f"{r['cd_dh']:7.1f}  {al_na*1e3:10.4f}  {al_dh*1e3:10.4f}  {ratio:6.2f}×")

# Best configuration
best_idx   = np.unravel_index(np.nanargmax(ratio_mat), ratio_mat.shape)
i_best, j_best = best_idx
r_best     = res_store[i_best][j_best]
L4_best    = L4_vals[i_best]
di_best    = di_cav_vals[j_best]
ratio_best = ratio_mat[i_best, j_best]

print(f"\n{'='*70}")
print(f"  OPTIMUM: L4={L4_best*1e3:.0f} mm,  di_cav={di_best*1e3:.0f} mm  "
      f"(wall={(D_main-di_best)/2*1e3:.1f} mm)")
print(f"  Improvement = {ratio_best:.2f}×   |   fn1={r_best['fn1']:.1f} Hz   "
      f"|   md={r_best['md']:.4f} kg   |   μ={r_best['mu']:.3f}")
print(f"  kd_DH={r_best['kd_dh']/1e6:.4f} MN/m   |   cd_DH={r_best['cd_dh']:.2f} N·s/m")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 8.  FULL STABILITY LOBES AT BEST, WORST AND REFERENCE DESIGNS
# ═══════════════════════════════════════════════════════════════════════════

worst_idx  = np.unravel_index(np.nanargmin(ratio_mat), ratio_mat.shape)
r_worst    = res_store[worst_idx[0]][worst_idx[1]]
# Reference: middle L4 and middle di_cav
r_ref      = res_store[N_L4//2][N_DI//2]

print("\nComputing full stability lobes …")
def get_lobes(r_, speed_dep):
    N_, a_, fc_ = stability_lobes(r_, speed_dep=speed_dep, wc_pts=4000, max_lobe=20)
    # no-absorber
    M2,D2,K2 = build_2x2(0.0, r_)
    axx,axy,ayx,ayy=AXX_UP,AXY_UP,AYX_UP,AYY_UP
    PSI_zf=r_['PSI_zf']
    wc_arr=np.linspace(0.3*r_['wn1'],2.5*r_['wn1'],3000)
    N_na=[]; a_na=[]; fc_na=[]
    for wc in wc_arr:
        fc=wc/(2*np.pi)
        hxx,hxy=frf_at_wc(wc,M2,D2,K2,PSI_zf); hyy=hxx; hyx=hxy
        h11=axx*hxx+axy*hyx; h12=axx*hxy+axy*hyy; h21=ayx*hxx+ayy*hyx; h22=ayx*hxy+ayy*hyy
        tr=h11+h22; dv=h11*h22-h12*h21
        if abs(dv)<1e-60: continue
        disc=tr**2-4*dv; sq=np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*dv),(-tr-sq)/(2*dv)]:
            rl=np.real(lam); il=np.imag(lam)
            if rl<0:
                al=-2*np.pi*rl/(Nt*Ktc)*(1+(il/rl)**2)
                if 0<al<0.020:
                    kp=il/rl; ep=np.pi-2*np.arctan(kp)
                    for k in range(20):
                        dn=k+ep/(2*np.pi)
                        if dn>0:
                            Np=60*fc/(Nt*dn)
                            if 100<=Np<=6100: N_na.append(Np); a_na.append(al*1000); fc_na.append(fc)
                break
    return (np.array(N_na),np.array(a_na),np.array(fc_na),
            N_,a_,fc_)

print("  Best design …")
N_na_b,a_na_b,_,N_dh_b,a_dh_b,_  = get_lobes(r_best,  speed_dep=True)
print("  Reference design …")
N_na_r,a_na_r,_,N_dh_r,a_dh_r,_  = get_lobes(r_ref,   speed_dep=True)
print("  Worst design …")
N_na_w,a_na_w,_,N_dh_w,a_dh_w,_  = get_lobes(r_worst, speed_dep=True)

# zero-speed lobes for best (comparison)
_,_,_,N_dh_b0,a_dh_b0,_ = get_lobes(r_best, speed_dep=False)

env_na_b  = lower_env(N_na_b, a_na_b, N_grid)
env_dh_b  = lower_env(N_dh_b, a_dh_b, N_grid)
env_dh_b0 = lower_env(N_dh_b0,a_dh_b0,N_grid)
env_na_r  = lower_env(N_na_r, a_na_r, N_grid)
env_dh_r  = lower_env(N_dh_r, a_dh_r, N_grid)
env_na_w  = lower_env(N_na_w, a_na_w, N_grid)
env_dh_w  = lower_env(N_dh_w, a_dh_w, N_grid)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 9.  INDIVIDUAL ROTATIONAL FORCE CONTRIBUTIONS AT BEST DESIGN
# ═══════════════════════════════════════════════════════════════════════════

def lobes_forces(r_, cor, gyro, cent, ri):
    """Stability lobes with selective rotational effects."""
    axx,axy,ayx,ayy=AXX_UP,AXY_UP,AYX_UP,AYY_UP
    PSI_zf=r_['PSI_zf']
    wn1=r_['wn1']
    Mm=r_['Mm']; Rii=r_['Rii'] if ri else 0.0
    Gii=r_['Gii']; Km=r_['Km']; Cm=r_['Cm_str']
    pzd=r_['PSI_zd']; kd=r_['kd_dh']; cd=r_['cd_dh']; md_=r_['md']

    def mat(Om):
        M11=Mm+Rii; K11=Km-(Om**2*Mm if cent else 0)
        Kabs=kd-(Om**2*md_ if cent else 0)
        Cof=(2*Om*Mm if cor else 0)-(2*Om*Gii if gyro else 0)
        Ca=(2*Om*md_ if cor else 0)
        M=np.diag([M11,md_,M11,md_])
        K=np.array([[K11+kd*pzd**2,-kd*pzd,0,0],[-kd*pzd,Kabs,0,0],
                    [0,0,K11+kd*pzd**2,-kd*pzd],[0,0,-kd*pzd,Kabs]])
        D=np.array([[Cm+cd*pzd**2,-cd*pzd,Cof,0],[-cd*pzd,cd,0,Ca],
                    [-Cof,0,Cm+cd*pzd**2,-cd*pzd],[0,-Ca,-cd*pzd,cd]])
        return M,D,K

    M0,D0,K0=mat(0.0)
    wc_arr=np.linspace(0.3*wn1,2.5*wn1,3000)
    N_o=[]; a_o=[]
    for wc in wc_arr:
        fc=wc/(2*np.pi)
        hxx,hxy=frf_at_wc(wc,M0,D0,K0,PSI_zf); hyy=hxx; hyx=hxy
        h11=axx*hxx+axy*hyx; h12=axx*hxy+axy*hyy; h21=ayx*hxx+ayy*hyx; h22=ayx*hxy+ayy*hyy
        tr=h11+h22; dv=h11*h22-h12*h21
        if abs(dv)<1e-60: continue
        disc=tr**2-4*dv; sq=np.sqrt(disc+0j)
        for lam in [(-tr+sq)/(2*dv),(-tr-sq)/(2*dv)]:
            rl=np.real(lam); il=np.imag(lam)
            if rl<0:
                al=-2*np.pi*rl/(Nt*Ktc)*(1+(il/rl)**2)
                if 0<al<0.020:
                    kp=il/rl; ep=np.pi-2*np.arctan(kp)
                    speed_used = cor or gyro or cent
                    for k in range(20):
                        dn=k+ep/(2*np.pi)
                        if dn>0:
                            Np=60*fc/(Nt*dn)
                            if 100<=Np<=6100:
                                if speed_used:
                                    Om=Np*2*np.pi/60; Mp,Dp,Kp=mat(Om)
                                    hxx2,hxy2=frf_at_wc(wc,Mp,Dp,Kp,PSI_zf)
                                    hyy2=hxx2; hyx2=hxy2
                                    h11_=axx*hxx2+axy*hyx2; h12_=axx*hxy2+axy*hyy2
                                    h21_=ayx*hxx2+ayy*hyx2; h22_=ayx*hxy2+ayy*hyy2
                                    tr_=h11_+h22_; dv_=h11_*h22_-h12_*h21_
                                    if abs(dv_)<1e-60: continue
                                    disc_=tr_**2-4*dv_; sq_=np.sqrt(disc_+0j)
                                    for lam_ in [(-tr_+sq_)/(2*dv_),(-tr_-sq_)/(2*dv_)]:
                                        rl_=np.real(lam_); il_=np.imag(lam_)
                                        if rl_<0:
                                            al_=-2*np.pi*rl_/(Nt*Ktc)*(1+(il_/rl_)**2)
                                            if al_>0:
                                                kap2=il_/rl_; eps2=np.pi-2*np.arctan(kap2)
                                                den2=k+eps2/(2*np.pi)
                                                if den2>0:
                                                    N2=60*fc/(Nt*den2)
                                                    if 100<=N2<=6100: N_o.append(N2); a_o.append(al_*1000)
                                            break
                                else:
                                    N_o.append(Np); a_o.append(al*1000)
                break
    return lower_env(np.array(N_o), np.array(a_o), N_grid)

print("  Computing force-effect lobes …")
env_cor  = lobes_forces(r_best, cor=True,  gyro=False, cent=False, ri=False)
env_gyro = lobes_forces(r_best, cor=False, gyro=True,  cent=False, ri=False)
env_cent = lobes_forces(r_best, cor=False, gyro=False, cent=True,  ri=False)
env_ri   = lobes_forces(r_best, cor=False, gyro=False, cent=False, ri=True)
print("  Done.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 10.  NATURAL FREQUENCY vs SPEED AT BEST DESIGN
# ═══════════════════════════════════════════════════════════════════════════

print("  Natural freqs vs speed …")
fw_abs, bw_abs, ab_abs = nat_freqs_vs_speed(r_best, speeds_spd)
# no-absorber version
r_noabs = dict(r_best)
r_noabs_2 = {**r_best, 'kd_dh':0.0, 'cd_dh':0.0, 'md':1e-6}
fw_na, bw_na, _ = nat_freqs_vs_speed(r_noabs_2, speeds_spd)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 11.  FRF AT SELECTED SPEEDS (best design)
# ═══════════════════════════════════════════════════════════════════════════

freqs_frf  = np.linspace(50, 2.5*r_best['fn1'], 1000)
speeds_frf = [0, 2000, 4000, 6000]
clrs_frf   = ["#333333","#1A80C4","#27AE60","#E77728"]
FRF_dir=[];  FRF_crs=[]
for rpm in speeds_frf:
    Om=rpm*2*np.pi/60; M4,D4,K4=build_4x4(Om,r_best)
    h_pairs=[frf_at_wc(2*np.pi*f,M4,D4,K4,r_best['PSI_zf']) for f in freqs_frf]
    FRF_dir.append(np.abs([h[0] for h in h_pairs]))
    FRF_crs.append(np.abs([h[1] for h in h_pairs]))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 12.  PLOTTING
# ═══════════════════════════════════════════════════════════════════════════

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.5,
    "figure.dpi": 150,
})

L4_mm  = L4_vals * 1e3
di_mm  = di_cav_vals * 1e3

In [ ]:
# ── FIGURE 1 ─ Heat maps and 3D surfaces (parameter space) ─────────────
fig1 = plt.figure(figsize=(18, 11))
fig1.suptitle(
    "Parameter 2 (Cavity Length L4) × Parameter 3 (Cavity Diameter di_cav)\n"
    "Rotary Boring Bar — All Rotational Effects Included",
    fontsize=11, fontweight="bold", y=1.00)
gs1 = gridspec.GridSpec(2, 4, figure=fig1, hspace=0.45, wspace=0.40,
                         left=0.05, right=0.97, top=0.91, bottom=0.07)

def heatmap(ax, data, title, cmap, fmt="{:.2f}", star=True):
    im = ax.imshow(data.T, origin='lower', aspect='auto', cmap=cmap,
                   extent=[L4_mm[0], L4_mm[-1], di_mm[0], di_mm[-1]])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for i in range(N_L4):
        for j in range(N_DI):
            if not np.isnan(data[i,j]):
                ax.text(L4_mm[i], di_mm[j], fmt.format(data[i,j]),
                        ha='center', va='center', fontsize=6.5,
                        color='white' if data[i,j]<np.nanmean(data)*1.2 else 'k')
    if star:
        ax.plot(L4_best*1e3, di_best*1e3, 'w*', ms=14, zorder=5, label='Optimum')
        ax.legend(fontsize=7)
    ax.set_xlabel("L4 [mm]"); ax.set_ylabel("di_cav [mm]")
    ax.set_title(title)

heatmap(fig1.add_subplot(gs1[0,0]), ratio_mat,    "Stability improvement ratio", 'YlOrRd')
heatmap(fig1.add_subplot(gs1[0,1]), alim_dh_mat,  "Min $a_{lim}$ with DH [mm]", 'Blues')
heatmap(fig1.add_subplot(gs1[0,2]), fn1_mat,      "$f_{n1}$ [Hz]",              'viridis',  fmt="{:.0f}")
heatmap(fig1.add_subplot(gs1[0,3]), mu_mat,       "Mass ratio $\\mu = m_d/M_m$",'plasma',   fmt="{:.2f}")
heatmap(fig1.add_subplot(gs1[1,0]), md_mat,       "Damper mass $m_d$ [kg]",     'copper',   fmt="{:.3f}")
heatmap(fig1.add_subplot(gs1[1,1]), PSI_mat,      r"Mode shape $\psi(z_d)$",    'RdYlGn',   fmt="{:.3f}")
heatmap(fig1.add_subplot(gs1[1,2]), kd_mat,       "$k_d^{DH}$ [MN/m]",         'cividis',  fmt="{:.2f}")
heatmap(fig1.add_subplot(gs1[1,3]), alim_na_mat,  "Min $a_{lim}$ no absorber [mm]",'Greys',fmt="{:.3f}")

plt.savefig(f"{SAVE}/boring_param23_heatmaps.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved → boring_param23_heatmaps.png")

In [ ]:
# ── FIGURE 2 ─ 3D surfaces ─────────────────────────────────────────────
fig2 = plt.figure(figsize=(18, 7))
fig2.suptitle("3D Parameter Surfaces — L4 × di_cav", fontsize=11, fontweight="bold")
L4M, DiM = np.meshgrid(L4_mm, di_mm, indexing='ij')

for idx, (data, title, cmap) in enumerate([
        (ratio_mat,   "Stability improvement ratio",  "YlOrRd"),
        (alim_dh_mat, "Min $a_{lim}$ DH [mm]",        "Blues"),
        (fn1_mat,     "1st natural freq [Hz]",         "viridis"),
]):
    ax3 = fig2.add_subplot(1, 3, idx+1, projection='3d')
    surf = ax3.plot_surface(L4M, DiM, np.where(np.isnan(data), np.nanmin(data), data),
                             cmap=cmap, edgecolor='none', alpha=0.85)
    ax3.scatter([L4_best*1e3], [di_best*1e3],
                [data[i_best,j_best] if not np.isnan(data[i_best,j_best]) else 0],
                color='red', s=80, zorder=5, label='Optimum')
    ax3.set_xlabel("L4 [mm]", labelpad=6)
    ax3.set_ylabel("di_cav [mm]", labelpad=6)
    ax3.set_title(title, pad=8)
    fig2.colorbar(surf, ax=ax3, fraction=0.03, pad=0.1)
    ax3.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f"{SAVE}/boring_param23_3D.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved → boring_param23_3D.png")

In [ ]:
# ── FIGURE 3 ─ Full stability lobe diagrams ────────────────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(15, 6))
fig3.suptitle(
    f"Stability Lobe Diagrams — Best, Reference and Worst Designs\n"
    f"Best: L4={L4_best*1e3:.0f}mm, di={di_best*1e3:.0f}mm "
    f"(ratio={ratio_best:.2f}×,  fn1={r_best['fn1']:.0f}Hz,  "
    f"md={r_best['md']:.3f}kg,  μ={r_best['mu']:.3f})",
    fontsize=10, fontweight="bold", y=1.01)

lobe_sets = [
    (env_na_b,  "No absorber (best)",        "#888", "-",  1.4),
    (env_dh_b0, "DH zero-speed (best)",      "#2166AC","--",1.6),
    (env_dh_b,  "DH speed-dep. (best)",      "#D6604D",":",  2.2),
    (env_na_r,  "No absorber (reference)",   "#AAA", "-",  1.0),
    (env_dh_r,  "DH speed-dep. (reference)", "#4DAC26","--",1.4),
    (env_na_w,  "No absorber (worst)",       "#CCC", "-",  0.8),
    (env_dh_w,  "DH speed-dep. (worst)",     "#9B59B6",":",  1.2),
]
for ax_s in axes3:
    for env, lbl, col, ls, lw in lobe_sets:
        ax_s.plot(N_grid, env, color=col, ls=ls, lw=lw, label=lbl)
    ax_s.set_xlim(0, 6000);  ax_s.set_ylim(0, None)
    ax_s.set_xlabel("Spindle speed [RPM]")
    ax_s.set_ylabel("$a_{lim}$ [mm]")
    ax_s.grid(True, alpha=0.3)
    ax_s.xaxis.set_major_locator(plt.MultipleLocator(1000))
    ax_s.xaxis.set_minor_locator(plt.MultipleLocator(500))

axes3[0].set_title("Full range 0–6000 RPM")
axes3[0].legend(fontsize=7.5, ncol=1)
axes3[1].set_xlim(200, 4000);  axes3[1].set_title("Zoomed 200–4000 RPM")
axes3[1].legend(fontsize=7.5)

plt.tight_layout()
plt.savefig(f"{SAVE}/boring_param23_lobes.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved → boring_param23_lobes.png")

In [ ]:
# ── FIGURE 4 ─ Force effects + FRF + nat-freq at best design ──────────
fig4 = plt.figure(figsize=(18, 10))
fig4.suptitle(
    f"Speed-Dependent Analysis at Optimal Design\n"
    f"L4={L4_best*1e3:.0f} mm,  di_cav={di_best*1e3:.0f} mm  "
    f"(wall={(D_main-di_best)/2*1e3:.0f} mm)  |  "
    f"fn1={r_best['fn1']:.0f} Hz  |  md={r_best['md']:.3f} kg  "
    f"|  μ={r_best['mu']:.3f}  |  kd={r_best['kd_dh']/1e6:.3f} MN/m",
    fontsize=10, fontweight="bold", y=1.01)
gs4 = gridspec.GridSpec(2, 3, figure=fig4, hspace=0.45, wspace=0.35,
                         left=0.06, right=0.97, top=0.91, bottom=0.08)

# 4a — stability lobes with individual force effects
ax4a = fig4.add_subplot(gs4[0, 0])
force_cfg = [
    (env_na_b,   "No absorber",              "#555","--",1.4),
    (env_cor,    "Coriolis only",            "#27AE60",":",1.6),
    (env_gyro,   "Gyroscopic only",          "#1A80C4","-.",1.5),
    (env_cent,   "Centrifugal only",         "#E77728","--",1.5),
    (env_ri,     "Rot. inertia only",        "#9B59B6",":",1.5),
    (env_dh_b,   "All effects (speed-dep.)", "#C0392B","-",2.2),
]
for env, lbl, col, ls, lw in force_cfg:
    ax4a.plot(N_grid, env, color=col, ls=ls, lw=lw, label=lbl)
ax4a.set_xlim(0, 6000);  ax4a.set_ylim(0, None)
ax4a.set_xlabel("Speed [RPM]");  ax4a.set_ylabel("$a_{lim}$ [mm]")
ax4a.set_title("Effect of individual\nrotational forces on stability")
ax4a.legend(fontsize=6.8);  ax4a.grid(True, alpha=0.3)

# 4b — natural frequencies vs speed
ax4b = fig4.add_subplot(gs4[0, 1])
ax4b.plot(speeds_spd, fw_abs, 'r-',  lw=1.6, label='Beam fwd (with absorber)')
ax4b.plot(speeds_spd, bw_abs, 'b-',  lw=1.6, label='Beam bwd (with absorber)')
ax4b.plot(speeds_spd, ab_abs, 'g--', lw=1.3, label='Absorber mode')
ax4b.plot(speeds_spd, fw_na,  'r:',  lw=1.0, alpha=0.6, label='Beam fwd (no absorber)')
ax4b.plot(speeds_spd, bw_na,  'b:',  lw=1.0, alpha=0.6, label='Beam bwd (no absorber)')
ax4b.set_xlabel("Speed [RPM]");  ax4b.set_ylabel("Natural frequency [Hz]")
ax4b.set_title("Natural frequencies vs speed\n(all rotational effects)")
ax4b.set_xlim(0, 6000);  ax4b.legend(fontsize=6.8, ncol=1);  ax4b.grid(True, alpha=0.3)

# 4c — mode shape with section shading
ax4c = fig4.add_subplot(gs4[0, 2])
z_arr = np.linspace(0, L_total, 400)
psi_v = np.array([r_best['psi'](z) for z in z_arr])
zb    = r_best['zb']
ax4c.fill_betweenx([-0.05, 1.05], zb[0]*1e3, zb[1]*1e3, alpha=0.15, color='cyan',  label='Sec1 flange')
ax4c.fill_betweenx([-0.05, 1.05], zb[1]*1e3, zb[2]*1e3, alpha=0.15, color='green', label='Sec2 taper')
ax4c.fill_betweenx([-0.05, 1.05], zb[2]*1e3, zb[3]*1e3, alpha=0.15, color='gray',  label='Sec3 hollow')
ax4c.fill_betweenx([-0.05, 1.05], zb[3]*1e3, zb[4]*1e3, alpha=0.25, color='orange',label='Sec4 cavity')
ax4c.fill_betweenx([-0.05, 1.05], zb[4]*1e3, L_total*1e3, alpha=0.15, color='red', label='Sec5 solid')
ax4c.plot(z_arr*1e3, psi_v, 'b-', lw=2.2, label=r'$\psi(z)$')
ax4c.axvline(r_best['zd']*1e3, color='k', ls='--', lw=1.2, label=f"zd={r_best['zd']*1e3:.0f}mm")
ax4c.set_xlabel("z [mm]");  ax4c.set_ylabel(r"$\psi(z)$ [norm.]")
ax4c.set_title(f"Mode shape at optimal design")
ax4c.legend(fontsize=6, loc='upper left');  ax4c.grid(True, alpha=0.3)

# 4d — Direct FRF vs speed
ax4d = fig4.add_subplot(gs4[1, 0])
for i, rpm in enumerate(speeds_frf):
    ax4d.semilogy(freqs_frf, FRF_dir[i], color=clrs_frf[i], lw=1.5, label=f'{rpm} RPM')
ax4d.set_xlabel("Frequency [Hz]");  ax4d.set_ylabel("|H_xx| [m/N]")
ax4d.set_title("Direct FRF |H_xx| vs speed\n(with DH absorber)")
ax4d.set_xlim(50, 2.5*r_best['fn1']);  ax4d.legend(fontsize=7)
ax4d.grid(True, which='both', alpha=0.3)

# 4e — Cross FRF (grows with speed due to Coriolis/Gyro)
ax4e = fig4.add_subplot(gs4[1, 1])
for i, rpm in enumerate(speeds_frf):
    ax4e.semilogy(freqs_frf, np.clip(FRF_crs[i],1e-14,None),
                  color=clrs_frf[i], lw=1.5, label=f'{rpm} RPM')
ax4e.set_xlabel("Frequency [Hz]");  ax4e.set_ylabel("|H_xy| [m/N]")
ax4e.set_title("Cross FRF |H_xy| vs speed\n(grows with speed: Coriolis+Gyro coupling)")
ax4e.set_xlim(50, 2.5*r_best['fn1']);  ax4e.legend(fontsize=7)
ax4e.grid(True, which='both', alpha=0.3)

# 4f — Parameter trade-off scatter
ax4f = fig4.add_subplot(gs4[1, 2])
sc = ax4f.scatter(mu_mat.ravel(), ratio_mat.ravel(),
                  c=fn1_mat.ravel(), cmap='plasma',
                  s=60, edgecolors='k', lw=0.4, zorder=3)
ax4f.scatter(r_best['mu'], ratio_best, s=200, marker='*', color='gold',
             edgecolors='k', lw=1, zorder=5, label='Optimum')
plt.colorbar(sc, ax=ax4f, label='fn1 [Hz]', fraction=0.04, pad=0.04)
ax4f.set_xlabel("Mass ratio μ = $m_d$ / $M_m$")
ax4f.set_ylabel("Stability improvement ratio")
ax4f.set_title("Trade-off: mass ratio vs stability\n(colour = fn1)")
ax4f.legend(fontsize=7.5);  ax4f.grid(True, alpha=0.3)

plt.savefig(f"{SAVE}/boring_param23_dynamics.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved → boring_param23_dynamics.png")

In [ ]:
# ── FIGURE 5 ─ 1D Sensitivity slices ──────────────────────────────────
fig5, axes5 = plt.subplots(2, 3, figsize=(16, 9))
fig5.suptitle(
    "1D Sensitivity: Varying each parameter independently\n"
    f"(one swept, the other fixed at optimal: L4={L4_best*1e3:.0f}mm, di_cav={di_best*1e3:.0f}mm)",
    fontsize=10, fontweight="bold")

# Slices along L4 (fixed di_best)
di_fixed_idx = j_best
L4_slice_fn1  = fn1_mat[:, di_fixed_idx]
L4_slice_mu   = mu_mat[:,  di_fixed_idx]
L4_slice_rat  = ratio_mat[:,di_fixed_idx]
L4_slice_al   = alim_dh_mat[:,di_fixed_idx]

ax = axes5[0,0]
ax.plot(L4_mm, L4_slice_fn1, 'b-o', ms=5); ax.axvline(L4_best*1e3,color='gold',ls='--',lw=1.5)
ax.set_xlabel("L4 [mm]"); ax.set_ylabel("fn1 [Hz]")
ax.set_title(f"fn1 vs L4 (di_cav={di_best*1e3:.0f}mm fixed)"); ax.grid(True,alpha=0.3)

ax = axes5[0,1]
ax.plot(L4_mm, L4_slice_mu, 'r-o', ms=5); ax.axvline(L4_best*1e3,color='gold',ls='--',lw=1.5)
ax.set_xlabel("L4 [mm]"); ax.set_ylabel("Mass ratio μ")
ax.set_title("μ vs L4"); ax.grid(True,alpha=0.3)

ax = axes5[0,2]
ax2=ax.twinx()
ax.plot(L4_mm, L4_slice_rat, 'b-o', ms=5, label='Improvement ratio')
ax2.plot(L4_mm, L4_slice_al, 'r-s', ms=5, label='Min alim_DH [mm]')
ax.axvline(L4_best*1e3,color='gold',ls='--',lw=1.5,label='Optimum')
ax.set_xlabel("L4 [mm]"); ax.set_ylabel("Improvement ratio",color='b')
ax2.set_ylabel("Min alim [mm]",color='r')
ax.set_title("Stability vs L4"); ax.grid(True,alpha=0.3)
l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2,lb1+lb2,fontsize=7)

# Slices along di_cav (fixed L4_best)
L4_fixed_idx = i_best
di_slice_fn1  = fn1_mat[L4_fixed_idx,:]
di_slice_mu   = mu_mat[L4_fixed_idx, :]
di_slice_rat  = ratio_mat[L4_fixed_idx,:]
di_slice_al   = alim_dh_mat[L4_fixed_idx,:]
wall_arr = (D_main - di_cav_vals)*1e3/2

ax = axes5[1,0]
ax.plot(di_mm, di_slice_fn1,'b-o',ms=5); ax.axvline(di_best*1e3,color='gold',ls='--',lw=1.5)
ax2=ax.twiny(); ax2.plot(wall_arr,di_slice_fn1,'g--',alpha=0)
ax2.set_xlabel("Wall thickness [mm]",color='green')
ax.set_xlabel("di_cav [mm]"); ax.set_ylabel("fn1 [Hz]")
ax.set_title(f"fn1 vs di_cav (L4={L4_best*1e3:.0f}mm fixed)"); ax.grid(True,alpha=0.3)

ax = axes5[1,1]
ax.plot(di_mm,di_slice_mu,'r-o',ms=5); ax.axvline(di_best*1e3,color='gold',ls='--',lw=1.5)
ax.set_xlabel("di_cav [mm]"); ax.set_ylabel("Mass ratio μ")
ax.set_title("μ vs di_cav"); ax.grid(True,alpha=0.3)

ax = axes5[1,2]
ax2=ax.twinx()
ax.plot(di_mm,di_slice_rat,'b-o',ms=5,label='Improvement ratio')
ax2.plot(di_mm,di_slice_al,'r-s',ms=5,label='Min alim_DH [mm]')
ax.axvline(di_best*1e3,color='gold',ls='--',lw=1.5,label='Optimum')
ax.set_xlabel("di_cav [mm]"); ax.set_ylabel("Improvement ratio",color='b')
ax2.set_ylabel("Min alim [mm]",color='r')
ax.set_title("Stability vs di_cav"); ax.grid(True,alpha=0.3)
l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2,lb1+lb2,fontsize=7)

plt.tight_layout()
plt.savefig(f"{SAVE}/boring_param23_sensitivity.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved → boring_param23_sensitivity.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 13.  FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*70)
print("SUMMARY OF OPTIMAL CONFIGURATION")
print("="*70)
print(f"  Parameter 1 (position) :  L3     = {L3_OPT*1e3:.0f} mm   (fixed)")
print(f"  Parameter 2 (length)   :  L4     = {L4_best*1e3:.0f} mm")
print(f"  Parameter 3 (diameter) :  di_cav = {di_best*1e3:.0f} mm  "
      f"(wall={(D_main-di_best)/2*1e3:.0f} mm)")
print(f"  L5 (solid tip)         :         {r_best['L5']*1e3:.0f} mm")
print()
print(f"  1st natural frequency  :  fn1  = {r_best['fn1']:.1f} Hz")
print(f"  Modal mass             :  Mm   = {r_best['Mm']:.4f} kg")
print(f"  Mode shape at absorber :  ψ(zd)= {r_best['PSI_zd']:.4f}")
print(f"  Absorber mass (WC)     :  md   = {r_best['md']:.4f} kg")
print(f"  Mass ratio             :  μ    = {r_best['mu']:.4f}")
print()
print(f"  Den Hartog kd          :  {r_best['kd_dh']/1e6:.4f} MN/m")
print(f"  Den Hartog cd          :  {r_best['cd_dh']:.2f} N·s/m  (ζ_dh={r_best['zeta_dh']:.3f})")
print()
print(f"  Min alim no absorber   :  {alim_na_mat[i_best,j_best]:.4f} mm")
print(f"  Min alim DH absorber   :  {alim_dh_mat[i_best,j_best]:.4f} mm")
print(f"  Stability improvement  :  {ratio_best:.2f}×")
print()
print("SAVED FIGURES:")
print("  boring_param23_heatmaps.png    — heat maps of all metrics")
print("  boring_param23_3D.png          — 3-D surfaces")
print("  boring_param23_lobes.png       — stability lobe diagrams")
print("  boring_param23_dynamics.png    — force effects, FRFs, nat-freqs")
print("  boring_param23_sensitivity.png — 1-D sensitivity slices")